## Part 1 – Project Setup & Environment

# Capstone Project

## Predicting Option Writing (Selling) Opportunities Using NSE F&O Data

### Notebook 01
## Data Ingestion & Master Dataset Creation

Author : Shripat Sharma

Program : Master of Artificial Intelligence & Machine Learning

Version : 1.0

Last Updated : July 2026

---

### Objective

The objective of this notebook is to create the master dataset for the project by:

- Reading NSE Bhavcopy CSV files
- Validating the data
- Loading the data into MariaDB
- Performing initial quality checks
- Creating database indexes
- Generating an ETL summary report

In [ ]:
#Install Required Libraries
#%pip install pandas numpy sqlalchemy pymysql tqdm pyarrow python-dotenv openpyxl matplotlib scikit-learn xgboost scipy statsmodels shap jupyterlab

In [1]:
import sys

print(sys.executable)
print(sys.version)

/opt/anaconda3/envs/capstone/bin/python
3.10.20 (main, Mar 11 2026, 17:43:48) [Clang 20.1.8 ]


In [2]:
# ==========================================
# Standard Libraries
# ==========================================

import os
import glob
import time
import warnings
from pathlib import Path
from datetime import datetime

# ==========================================
# Data Libraries
# ==========================================

import numpy as np
import pandas as pd

# ==========================================
# Database Libraries
# ==========================================

from sqlalchemy import create_engine
from sqlalchemy import text

import pymysql

# ==========================================
# Progress Bar
# ==========================================

from tqdm import tqdm

# ==========================================
# Display Settings
# ==========================================

from IPython.display import display

# ==========================================
# Ignore Warnings
# ==========================================

warnings.filterwarnings("ignore")

In [3]:
# Display Settings

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 100)

In [4]:
from pathlib import Path

PROJECT_ROOT = Path("/Users/shripat/Walsh/Term3/Bhavco")

DATA_ROOT = PROJECT_ROOT / "data"

RAW_DATA_PATH = DATA_ROOT / "raw"

# Individual data sources
BHAVCOPY_PATH = RAW_DATA_PATH / "bhavcopy"

VIX_PATH = RAW_DATA_PATH / "india_vix"

TBILL_PATH = RAW_DATA_PATH / "risk_free_rate"

EXPIRY_PATH = RAW_DATA_PATH / "expiry_calendar"

LOTSIZE_PATH = RAW_DATA_PATH / "lot_size"

PROCESSED_PATH = DATA_ROOT / "processed"

PARQUET_PATH = DATA_ROOT / "parquet"

FINAL_PATH = DATA_ROOT / "final"

In [5]:
paths = {
    "Bhavcopy": BHAVCOPY_PATH,
    "India VIX": VIX_PATH,
    "T-Bill": TBILL_PATH,
    "Processed": PROCESSED_PATH,
    "Parquet": PARQUET_PATH
}

for name, path in paths.items():
    print(f"{name:15} : {path}")
    print(f"Exists         : {path.exists()}")
    print("-" * 60)

Bhavcopy        : /Users/shripat/Walsh/Term3/Bhavco/data/raw/bhavcopy
Exists         : True
------------------------------------------------------------
India VIX       : /Users/shripat/Walsh/Term3/Bhavco/data/raw/india_vix
Exists         : True
------------------------------------------------------------
T-Bill          : /Users/shripat/Walsh/Term3/Bhavco/data/raw/risk_free_rate
Exists         : True
------------------------------------------------------------
Processed       : /Users/shripat/Walsh/Term3/Bhavco/data/processed
Exists         : True
------------------------------------------------------------
Parquet         : /Users/shripat/Walsh/Term3/Bhavco/data/parquet
Exists         : True
------------------------------------------------------------


In [6]:
csv_files = list(BHAVCOPY_PATH.rglob("*.csv"))

print(f"Total CSV Files Found : {len(csv_files)}")

Total CSV Files Found : 13


In [7]:
sample = pd.DataFrame({

    "CSV Files":

    [str(file.relative_to(PROJECT_ROOT))

     for file in csv_files[:10]]

})

display(sample)

,CSV Files
0,data/raw/bhavcopy/FO_2026_Part4.csv
1,data/raw/bhavcopy/FO_2026_Part5.csv
2,data/raw/bhavcopy/FO_2026_Part2.csv
3,data/raw/bhavcopy/FO_2026_Part3.csv
4,data/raw/bhavcopy/FO_2026_Part1.csv
5,data/raw/bhavcopy/FO_2025_Part8.csv
6,data/raw/bhavcopy/FO_2025_Part5.csv
7,data/raw/bhavcopy/FO_2025_Part4.csv
8,data/raw/bhavcopy/FO_2025_Part6.csv
9,data/raw/bhavcopy/FO_2025_Part7.csv
